# Module 5 — Exporting at Scale & Harmonic Regression Baselines

**Goal:** learn asynchronous batch exports (the real way to pull down state-scale results), and build a per-pixel/per-polygon harmonic regression seasonal baseline — the statistical backbone of a change-point-based glyphosate detector.

**Time:** ~2 hours


In [ ]:
import ee
try:
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')

import pandas as pd


### Batch exports: `ee.batch.Export`

`.getInfo()` is synchronous and size-limited — fine for a few thousand rows, not for a state-wide year of COMTRS chunks. `Export.table.toDrive` (or `toCloudStorage`/`toAsset`) instead submits an asynchronous **task** that runs on Google's infrastructure and writes a file when done. You poll task status instead of blocking on a response.


In [ ]:
point = ee.Geometry.Point(-121.5, 38.5)
demo_fc = ee.FeatureCollection([ee.Feature(point, {'label': 'demo'})])

task = ee.batch.Export.table.toDrive(
    collection=demo_fc,
    description='demo_export',
    folder='gee_exports',
    fileFormat='CSV'
)
task.start()
print(task.status())


In [ ]:
# Poll (in a real script you'd sleep/loop; here's the one-shot check)
import time
# for _ in range(5):
#     status = task.status()
#     print(status['state'])
#     if status['state'] in ('COMPLETED', 'FAILED', 'CANCELLED'):
#         break
#     time.sleep(10)


### Exercise 5.1
Take your `extract_zonal_timeseries`-style pipeline from Module 4, but instead of `.getInfo()`, wrap the final long-format `FeatureCollection` in an `Export.table.toDrive` call. Think about (write as a comment, doesn't need to run): for a full year × full state, would you submit **one giant export** or split into multiple tasks (e.g. by month, or by spatial tile)? What are the tradeoffs (task limits, retry cost if one fails, how you'd merge outputs afterward)?


In [ ]:
# TODO: your code / notes here


---
## Harmonic regression — modeling the seasonal baseline

A vegetation index time series is roughly periodic across a year (green-up, peak, senescence). A **harmonic regression** fits that seasonal cycle as a sum of sine/cosine terms plus a trend, giving you an *expected* NDVI for any date. The residual — observed minus expected — is what you scan for anomalies (like a glyphosate-induced dip that a pure calendar-based threshold would miss, since "normal" NDVI itself varies by season).

The model, per pixel/polygon:

`NDVI(t) ≈ a + b·t + c·cos(2πt) + d·sin(2πt)  [+ higher harmonics if needed]`

where `t` is time in **fractional years**. This is linear in its coefficients, so — conveniently — GEE has a built-in linear regression reducer you can run per-pixel across an `ImageCollection`, entirely server-side.


In [ ]:
def add_harmonic_bands(image):
    date = ee.Date(image.get('system:time_start'))
    years = date.difference(ee.Date('2020-01-01'), 'year')
    time_radians = ee.Image(years.multiply(2 * 3.141592653589793)).rename('t')
    return image.addBands(time_radians).addBands(time_radians.cos().rename('cos')) \
                 .addBands(time_radians.sin().rename('sin')) \
                 .addBands(ee.Image.constant(1).rename('constant'))

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_ndvi(image):
    return image.addBands(image.normalizedDifference(['B8', 'B4']).rename('NDVI'))

point = ee.Geometry.Point(-121.5, 38.5)
aoi = point.buffer(500)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterDate('2019-01-01', '2022-01-01')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))
      .map(mask_s2_clouds)
      .map(add_ndvi)
      .map(add_harmonic_bands))

indep_vars = ['constant', 't', 'cos', 'sin']
dep_var = 'NDVI'

harmonic_regression = s2.select(indep_vars + [dep_var]).reduce(
    ee.Reducer.linearRegression(numX=len(indep_vars), numY=1)
)
print(harmonic_regression.bandNames().getInfo())


The `coefficients` band from `linearRegression` holds `[constant, t, cos, sin]` weights per pixel. To get a *predicted* NDVI for any given image, multiply that image's `[constant, t, cos, sin]` bands by the fitted coefficients and sum.


In [ ]:
coeffs = harmonic_regression.select('coefficients')

def predict_ndvi(image):
    pred = image.select(indep_vars).multiply(coeffs.arrayProject([0]).arrayFlatten([indep_vars])) \
                 .reduce('sum').rename('predicted_NDVI')
    residual = image.select('NDVI').subtract(pred).rename('residual')
    return image.addBands(pred).addBands(residual)

s2_with_predictions = s2.map(predict_ndvi)

# Pull the observed/predicted/residual series at our point into pandas
def extract(image):
    vals = image.select(['NDVI', 'predicted_NDVI', 'residual']).reduceRegion(ee.Reducer.mean(), point, 10)
    return ee.Feature(None, vals.set('date', image.date().format('YYYY-MM-dd')))

records = s2_with_predictions.map(extract).getInfo()['features']
df = pd.DataFrame([r['properties'] for r in records]).dropna()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.plot(x='date', y=['NDVI', 'predicted_NDVI'], figsize=(11,4))


In [ ]:
df.plot(x='date', y='residual', figsize=(11,3), title='Residual (observed - predicted)')


### Exercise 5.2
A dip that lasts only 4-5 days (the short end of glyphosate's observable window, per your project notes) can be easy to miss if your revisit frequency is coarse and your residual threshold is too strict. Try: (a) plot residual vs. a rolling z-score of residual (mean/std of past N residuals) instead of raw residual — does that make short dips more detectable against normal seasonal noise? (b) note what revisit cadence (Sentinel-2: ~5 days at the equator, less frequent with clouds) implies about your minimum detectable event duration.


In [ ]:
# TODO: your code / notes here


### Project 5 — "Baseline + residual pipeline for one polygon"

Wrap the harmonic-regression pattern above into a function:

```python
def fit_harmonic_baseline(geometry, start, end, collection_id='COPERNICUS/S2_SR_HARMONIZED') -> pd.DataFrame
```

returning a DataFrame with `date`, `NDVI`, `predicted_NDVI`, `residual` for a given AOI. This — run per COMTRS chunk instead of a single point — is your multi-year "expectation model" that flags candidate spray events as residual dips, which is exactly the recall-first candidate generator your project notes describe. Note in a markdown cell: since spray events themselves could contaminate the harmonic fit if included in the training window (your own earlier concern), what's one way you'd guard against that — e.g. fitting on multiple years and down-weighting outliers, or an iterative refit that excludes points already flagged as anomalies?


In [ ]:
def fit_harmonic_baseline(geometry, start, end, collection_id='COPERNICUS/S2_SR_HARMONIZED'):
    # TODO
    pass


---
**Checkpoint:** you should understand both halves of the change-detection idea now — (1) fit a per-location expected seasonal curve with harmonic regression, and (2) treat large residuals as candidate events. Final module: putting fundamentals + zonal extraction + baseline residuals together into one small end-to-end capstone.